# 01 · Assemble the platform
*Hill requisitions the base.*

`azd up` already put the Azure parts in place: the workspace, a storage account and an access connector. They are empty buildings.
This notebook furnishes them with **Unity Catalog** objects: an external location, the `shield` catalog, five schemas and three volumes. Every later chapter lives inside these.

| Concept | Where you meet it |
|---|---|
| How storage credential, external location, catalog, schema and volume fit together | the next section |
| External location (a path plus a credential) | Step 1 |
| Catalog with a managed location | Step 2 |
| Schemas: one per medallion layer, plus landing and ops | Step 3 |
| Managed volumes for files | Step 4 |

Deep dive and the manual steps: [docs/01-assemble-the-platform.md](../../docs/01-assemble-the-platform.md)

**Needs:**
* `azd up` finished (it outputs `STORAGE_ACCOUNT_NAME`), and you did the one manual step from the doc: a **storage credential** named `shield_cred` that wraps the access connector. The setup cell stops with a message if it cannot find it.
* Privileges: `CREATE EXTERNAL LOCATION` on the metastore and on the credential, and `CREATE CATALOG` on the metastore. A metastore admin has them. Without them, Step 1 or Step 2 fails with a permission error; ask a metastore admin.
* Any compute; serverless is fine. No earlier chapter is needed besides chapter 00 as a sanity check.

This notebook changes workspace state (Unity Catalog objects, not Azure resources). Every statement uses `IF NOT EXISTS`, so re-running is safe, see the Clean up section for what exists afterwards.
The **Expected** values are expectations until you have run the notebook in your workspace.

## How the platform fits together

**The idea:** Unity Catalog never lets code touch cloud storage directly. It puts a chain of named objects in between, and permissions are granted on those objects. It is a product feature, but the names of the schemas and the way they map to layers are this repo's own convention.

```
  Azure storage account  (containers: uc-managed, landing)          created by azd up
          ▲  reads and writes through
  Access connector (a managed identity)                             created by azd up
          ▲  wrapped by
  STORAGE CREDENTIAL  shield_cred                                    created by you, once, by hand
          ▲  used by
  EXTERNAL LOCATION   shield_managed  = abfss://uc-managed@<account>/   Step 1
          ▲  gives storage to
  CATALOG             shield          (MANAGED LOCATION .../shield)      Step 2
     ├─ SCHEMA landing    ─ volumes: staging, raw, checkpoints           Steps 3 and 4
     ├─ SCHEMA bronze / silver / gold   (tables, from chapter 03 on)    Step 3
     └─ SCHEMA ops                                                     Step 3
```

**How this notebook implements it**

* **External location** `shield_managed` points at the `uc-managed` container and uses the credential `shield_cred`.
* **Catalog** `shield` has `MANAGED LOCATION` set to the folder `shield` inside that container. Managed tables and managed volumes of the catalog are stored there, and you never see the file paths.
* **Schemas** `bronze`, `silver` and `gold` will hold the three medallion layers (explained in chapter 05). `landing` holds files, `ops` holds operational tables used later.
* **Volumes** are Unity Catalog's folders for files. `staging` receives the generated seed data (chapter 02), `raw` is where files "arrive" (chapter 03), `checkpoints` holds the streaming state of Auto Loader. They are **managed** volumes: Unity Catalog decides where the files live, and dropping the volume deletes the files.

**Why it helps:** access is governed in one place (grants on catalog, schema or volume, chapter 11) and no storage keys are involved. The storage account has shared-key access disabled, so everything goes through Entra identities.

**Not covered yet:** granting access to other people (chapter 11), an *external* volume on the `landing` container, which this notebook deliberately leaves unused (mini-project in the doc), and the layers themselves (chapter 05).

## Setup

Two widgets, because the storage account name is different for everybody. Fill in `storage_account` (from `azd env get-values`, `STORAGE_ACCOUNT_NAME`) and leave `credential` as it is unless you named the credential differently.

In [ ]:
def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


dbutils.widgets.text("storage_account", "", "Storage account name (azd output STORAGE_ACCOUNT_NAME)")
dbutils.widgets.text("credential", "shield_cred", "Storage credential name")
storage_account = dbutils.widgets.get("storage_account")
credential = dbutils.widgets.get("credential")
assert storage_account, "Fill in the storage_account widget"

credentials = {row["name"] for row in spark.sql("SHOW STORAGE CREDENTIALS").collect()}
assert credential in credentials, (
    f"Storage credential '{credential}' not found (visible to you: {sorted(credentials)}). "
    "Create it first, see docs/01-assemble-the-platform.md."
)

## 1 · From storage to catalog

### Step 1 · External location
**Goal:** tell Unity Catalog which container it may use, and with which credential.

| Input | Value |
|---|---|
| Container | `uc-managed` in your storage account |
| Credential | `shield_cred` |

**Expected:** the statement succeeds and `SHOW EXTERNAL LOCATIONS` lists `shield_managed` with the URL of the `uc-managed` container (the metastore may hold other locations too).

What you will see:

| name | url | comment |
|---|---|---|
| shield_managed | abfss://uc-managed@`<your account>`.dfs.core.windows.net/ | |

Safe to re-run. `IF NOT EXISTS` does nothing when the name is taken, so a location created earlier with a wrong account name stays wrong: drop it with `DROP EXTERNAL LOCATION shield_managed` and run the cell again.

In [ ]:
managed_url = f"abfss://uc-managed@{storage_account}.dfs.core.windows.net/"

spark.sql(
    f"""
    CREATE EXTERNAL LOCATION IF NOT EXISTS shield_managed
    URL '{managed_url}'
    WITH (STORAGE CREDENTIAL `{credential}`)
    """
)

external_locations = spark.sql("SHOW EXTERNAL LOCATIONS")
display(external_locations)

location_urls = {row["name"]: row["url"] for row in external_locations.collect()}
show_checks([
    ("external location shield_managed exists", "shield_managed" in location_urls, True),
    ("its URL", location_urls.get("shield_managed", "").rstrip("/"), managed_url.rstrip("/")),
])

### Step 2 · Catalog
**Goal:** create the catalog `shield` and give it storage of its own.

**Expected:** `shield` appears in `SHOW CATALOGS`. `DESCRIBE CATALOG EXTENDED shield` shows a storage root that ends in `uc-managed@<your account>.dfs.core.windows.net/shield` (the label of that row can differ between runtime versions, so look for the path).

Why `MANAGED LOCATION` is needed: the regional metastore that Azure creates for the workspace has no storage of its own, so a catalog must say where its managed tables and volumes live. The location has to be covered by an external location, which is what Step 1 made. You own that external location, so you have the privilege to put a catalog on it.

Safe to re-run (`IF NOT EXISTS`). The catalog keeps the managed location it was first created with.

In [ ]:
spark.sql(
    f"""
    CREATE CATALOG IF NOT EXISTS shield
    MANAGED LOCATION 'abfss://uc-managed@{storage_account}.dfs.core.windows.net/shield'
    COMMENT 'S.H.I.E.L.D. intelligence platform'
    """
)

display(spark.sql("DESCRIBE CATALOG EXTENDED shield"))

catalogs = {row[0] for row in spark.sql("SHOW CATALOGS").collect()}
show_checks([("catalog shield exists", "shield" in catalogs, True)])

## 2 · Rooms inside the catalog

### Step 3 · Schemas
**Goal:** create the five schemas, one per layer plus `landing` for files and `ops` for operational tables.

**Expected:** `SHOW SCHEMAS IN shield` lists `landing`, `bronze`, `silver`, `gold` and `ops`. Unity Catalog also shows `default` and `information_schema`, which it creates itself.

What you will see (order may differ):

| databaseName |
|---|
| bronze |
| default |
| gold |
| information_schema |
| landing |
| ops |
| silver |

Safe to re-run (`IF NOT EXISTS`).

In [ ]:
wanted_schemas = ["landing", "bronze", "silver", "gold", "ops"]
for schema in wanted_schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS shield.{schema}")

schemas = spark.sql("SHOW SCHEMAS IN shield")
display(schemas)

existing_schemas = {row[0] for row in schemas.collect()}
show_checks([(f"schema shield.{schema} exists", schema in existing_schemas, True) for schema in wanted_schemas])

### Step 4 · Volumes
**Goal:** create the three folders for files that chapters 02 and 03 use.

| Volume | Holds | Used by |
|---|---|---|
| `shield.landing.staging` | the generated seed data | chapter 02 writes, chapter 03 reads |
| `shield.landing.raw` | files "arriving from the field" | chapter 03 |
| `shield.landing.checkpoints` | Auto Loader checkpoints and inferred schemas | chapter 03 |

**Expected:** three volumes in `shield.landing`, all of type `MANAGED`. From now on their files are ordinary paths, for example `/Volumes/shield/landing/staging`.

Safe to re-run (`IF NOT EXISTS`).

In [ ]:
spark.sql("CREATE VOLUME IF NOT EXISTS shield.landing.staging COMMENT 'Generated seed data'")
spark.sql("CREATE VOLUME IF NOT EXISTS shield.landing.raw COMMENT 'Files arriving from the field'")
spark.sql("CREATE VOLUME IF NOT EXISTS shield.landing.checkpoints COMMENT 'Streaming checkpoints and schemas'")

landing_volumes = spark.sql(
    """
    SELECT volume_name, volume_type, comment
    FROM shield.information_schema.volumes
    WHERE volume_schema = 'landing'
    ORDER BY volume_name
    """
)
display(landing_volumes)

volume_rows = landing_volumes.collect()
show_checks([
    ("volumes in shield.landing", [row["volume_name"] for row in volume_rows], ["checkpoints", "raw", "staging"]),
    ("volume types", sorted({row["volume_type"] for row in volume_rows}), ["MANAGED"]),
])

## 👀 In the workspace

* **Catalog** (sidebar) → `shield`: the five schemas. Open `landing` → **Volumes** to see the three volumes.
* **Catalog → Connect → External locations**: `shield_managed`. Its **Test connection** button checks the credential against the container.
* **Catalog → Connect → Credentials**: `shield_cred`.

## 🧹 Clean up

Nothing is deleted: every later chapter needs this catalog. What this notebook created, all inside Unity Catalog and none of it in Azure: the external location `shield_managed`, the catalog `shield`, five schemas and three managed volumes. The credential `shield_cred` you made by hand.

`azd down --purge` removes the Azure resources. The Unity Catalog objects above belong to the regional metastore, which `azd` does not manage, so I expect them to survive that (not verified): drop them yourself if you want a clean slate. To start over, run the commented cell below. **It deletes every table and file in `shield`**, including the data of later chapters.

In [ ]:
# DROP CATALOG ... CASCADE also deletes the managed tables and managed volumes inside the catalog.
# spark.sql("DROP CATALOG IF EXISTS shield CASCADE")
# spark.sql("DROP EXTERNAL LOCATION IF EXISTS shield_managed")

## 🎯 Recap

| Object | Name here | Made by |
|---|---|---|
| Storage credential | `shield_cred` | you, by hand (once) |
| External location | `shield_managed` | Step 1 |
| Catalog | `shield` (managed location `uc-managed/shield`) | Step 2 |
| Schemas | `landing`, `bronze`, `silver`, `gold`, `ops` | Step 3 |
| Managed volumes | `staging`, `raw`, `checkpoints` in `shield.landing` | Step 4 |

Unity Catalog stands between your code and the storage: credential, then location, then catalog, then the objects you query.

Try-it-yourself mini-projects: [docs/01-assemble-the-platform.md](../../docs/01-assemble-the-platform.md#try-it-yourself)

> 💡 Next: chapter 02 generates the seed data into `shield.landing.staging` and explores it with DataFrames and SQL.